In [1]:
#6--Text Spam Classification Using TF-IDF and SVM yeayyy
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

# Given dataset from the question
emails = [
    "win a free smartphone today.",
    "Your account statement is available.",
    "Claim your exclusive cash reward now.",
    "Please review the attached project report.",
    "Congratulations! You have won a free vacation.",
    "The examination schedule has been announced.",
    "Earn money quickly with this special offer.",
    "The faculty meeting starts at 11 AM.",
    "You have been selected for a free gift.",
    "Submit your laboratory assignment by Monday.",
    "Get a discount on your next purchase.",
    "Your internship interview is scheduled for Friday.",
]

labels = [
    "spam",
    "ham",
    "spam",
    "ham",
    "spam",
    "ham",
    "spam",
    "ham",
    "spam",
    "ham",
    "spam",
    "ham",
]

test_emails = [
    "Congratulations, claim your free prize now!",
    "Please attend the project meeting tomorrow.",
]

# -------------------------------------------------------------
# Task a): Preprocess the text and split the dataset into training and testing sets 
# -------------------------------------------------------------
X_train, X_test_split, y_train, y_test_split = train_test_split(
    emails, labels, test_size=0.25, random_state=42
)

# -------------------------------------------------------------
# Task b): Apply TF-IDF feature extraction using TfidfVectorizer
# -------------------------------------------------------------
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test_split)
X_external_tfidf = vectorizer.transform(test_emails)

# -------------------------------------------------------------
# Task c): Train a Support Vector Machine (SVM) classifier using SVC
# -------------------------------------------------------------
svm_model = SVC(kernel="linear")
svm_model.fit(X_train_tfidf, y_train)

# -------------------------------------------------------------
# Task d): Predict whether the test emails are spam or ham
# -------------------------------------------------------------
svm_preds = svm_model.predict(X_external_tfidf)
print("SVM Predictions on Test Emails:", svm_preds)

# -------------------------------------------------------------
# Task e): Display the predicted class and decision score for each test email
# -------------------------------------------------------------
svm_decision = svm_model.decision_function(X_external_tfidf)

print("\n--- SVM Test Email Results ---")
for email, pred, score in zip(test_emails, svm_preds, svm_decision):
  print(f"Email: '{email}' | Predicted: {pred} | Decision Score: {score:.4f}")

# -------------------------------------------------------------
# Task f): Evaluate the model using accuracy, precision, recall, F1-score, and confusion matrix
# -------------------------------------------------------------
y_pred_svm = svm_model.predict(X_test_tfidf)

print("\n--- SVM Evaluation Metrics ---")
print("Accuracy:", accuracy_score(y_test_split, y_pred_svm))
print("Precision:", precision_score(y_test_split, y_pred_svm, pos_label="spam"))
print("Recall:", recall_score(y_test_split, y_pred_svm, pos_label="spam"))
print("F1-Score:", f1_score(y_test_split, y_pred_svm, pos_label="spam"))
print("Confusion Matrix:\n", confusion_matrix(y_test_split, y_pred_svm))

# -------------------------------------------------------------
# Task g): Compare the performance of the SVM classifier with the Multinomial Naive Bayes classifier
# -------------------------------------------------------------
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

print("\n--- Multinomial Naive Bayes Comparison ---")
print("Naive Bayes Accuracy:", accuracy_score(y_test_split, y_pred_nb))
print(
    "Naive Bayes F1-Score:", f1_score(y_test_split, y_pred_nb, pos_label="spam")
)
print("Confusion Matrix (NB):\n", confusion_matrix(y_test_split, y_pred_nb))

SVM Predictions on Test Emails: ['spam' 'ham']

--- SVM Test Email Results ---
Email: 'Congratulations, claim your free prize now!' | Predicted: spam | Decision Score: 0.5943
Email: 'Please attend the project meeting tomorrow.' | Predicted: ham | Decision Score: -0.8885

--- SVM Evaluation Metrics ---
Accuracy: 0.6666666666666666
Precision: 1.0
Recall: 0.5
F1-Score: 0.6666666666666666
Confusion Matrix:
 [[1 0]
 [1 1]]

--- Multinomial Naive Bayes Comparison ---
Naive Bayes Accuracy: 0.6666666666666666
Naive Bayes F1-Score: 0.6666666666666666
Confusion Matrix (NB):
 [[1 0]
 [1 1]]


In [2]:
#7-->Binary Sentiment Classification Using a Simple RNN  #dogla RNN
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Given dataset from the question 
texts = [
    "I love this movie",
    "The film was absolutely wonderful",
    "This movie is fantastic",
    "I hate this film",
    "The story was boring",
    "The movie was disappointing",
    "An excellent and enjoyable film",
    "The acting was terrible"
]

labels = [1, 1, 1, 0, 0, 0, 1, 0]

test_texts = [
    "I really love this film",
    "The movie was boring and terrible"
]

# -------------------------------------------------------------
# Task a): Tokenize and convert into integer sequences
# -------------------------------------------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

# Convert texts to sequences of integers
train_sequences = tokenizer.texts_to_sequences(texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

print("Train Integer Sequences:", train_sequences)
print("Test Integer Sequences:", test_sequences)

# -------------------------------------------------------------
# Task b): Apply padding using pad_sequences
# -------------------------------------------------------------
# Find maximum sequence length (or you can specify a fixed length like maxlen=5)
max_length = max(max(len(seq) for seq in train_sequences), # avu na karvu hoy toh u can directly write 6, 7 or a avg num
                 max(len(seq) for seq in test_sequences))

# Pad sequences (pre-padding is standard for RNNs)
X_train = pad_sequences(train_sequences, maxlen=max_length, padding='pre') 
X_test = pad_sequences(test_sequences, maxlen=max_length, padding='pre')

print("\nPadded Train Data:\n", X_train)
print("\nPadded Test Data:\n", X_test)

import numpy as np
from tensorflow.keras.layers import Dense, Embedding, SimpleRNN   # DL once more (●'◡'●) 
from tensorflow.keras.models import Sequential

# -------------------------------------------------------------
# Task c): Build a Simple RNN model
# -------------------------------------------------------------
vocab_size = len(tokenizer.word_index) + 1  # Total unique words + 1 for padding   (basically the input numbers)

model = Sequential(
    [
        Embedding(
            input_dim=vocab_size, output_dim=16, input_length=max_length   #to convert to vectors can skip input_length totally
        ),
        SimpleRNN(32),
        Dense(1, activation="sigmoid"),
    ]
)

model.summary()

# -------------------------------------------------------------
# Task d): Compile and train the model
# -------------------------------------------------------------
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

# Convert labels to numpy arrays # we dont convert x_train to np array as it was already coverted while pad_sequences
y_train = np.array(labels)

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=2, verbose=1)

# -------------------------------------------------------------
# Task e): Predict the sentiment probabilities for test sentences
# -------------------------------------------------------------
predicted_probabilities = model.predict(X_test)
print("\nPredicted Probabilities:\n", predicted_probabilities)

# -------------------------------------------------------------
# Task f): Display predicted probability and classify (Threshold = 0.5)
# -------------------------------------------------------------
for text, prob in zip(test_texts, predicted_probabilities):
  sentiment = "Positive" if prob[0] >= 0.5 else "Negative"
  print(
      f"Text: '{text}' | Probability: {prob[0]:.4f} | Classification:"
      f" {sentiment}"
  )

# -------------------------------------------------------------
# Task g): Explain the role of Embedding and SimpleRNN layers
# -------------------------------------------------------------
# Write these points in your exam if asked for a theoretical explanation:
#
# 1. Embedding Layer: Converts sparse integer-encoded word indices into dense
#    vectors of fixed size (embeddings). It captures semantic meanings and
#    relationships between words in a continuous vector space.
#
# 2. SimpleRNN Layer: Processes the sequence of vectors step-by-step while
#    maintaining a hidden state (memory) to capture contextual dependencies
#    across words in the text.


Train Integer Sequences: [[6, 7, 3, 4], [1, 5, 2, 8, 9], [3, 4, 10, 11], [6, 12, 3, 5], [1, 13, 2, 14], [1, 4, 2, 15], [16, 17, 18, 19, 5], [1, 20, 2, 21]]
Test Integer Sequences: [[6, 7, 3, 5], [1, 4, 2, 14, 18, 21]]

Padded Train Data:
 [[ 0  0  6  7  3  4]
 [ 0  1  5  2  8  9]
 [ 0  0  3  4 10 11]
 [ 0  0  6 12  3  5]
 [ 0  0  1 13  2 14]
 [ 0  0  1  4  2 15]
 [ 0 16 17 18 19  5]
 [ 0  0  1 20  2 21]]

Padded Test Data:
 [[ 0  0  6  7  3  5]
 [ 1  4  2 14 18 21]]

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 6, 16)             352       
                                                                 
 simple_rnn (SimpleRNN)      (None, 32)                1568      
                                                                 
 dense (Dense)               (None, 1)                 33        
                                             

In [3]:
#8 =>Book Review Classification Using an LSTM Model 
import numpy as np
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Given dataset from the question
reviews = [
    "This book is engaging and beautifully written.",
    "I thoroughly enjoyed reading this book.",
    "The characters and storyline are excellent.",
    "This is one of the best books I have read.",
    "The story is dull and poorly written.",
    "I found the book boring and disappointing.",
    "The plot is confusing and the ending is terrible.",
    "I would not recommend this book.",
]

labels = [1, 1, 1, 1, 0, 0, 0, 0]

test_reviews = [
    "The book was beautifully written and enjoyable.",
    "The story was dull and disappointing.",
]

# -------------------------------------------------------------
# Task a): Tokenize and convert into padded integer sequences
# -------------------------------------------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts(reviews)

train_sequences = tokenizer.texts_to_sequences(reviews)
test_sequences = tokenizer.texts_to_sequences(test_reviews)

max_length = max(
    max(len(seq) for seq in train_sequences),
    max(len(seq) for seq in test_sequences),
)

X_train = pad_sequences(train_sequences, maxlen=max_length, padding="pre")
X_test = pad_sequences(test_sequences, maxlen=max_length, padding="pre")

y_train = np.array(labels)
vocab_size = len(tokenizer.word_index) + 1

# -------------------------------------------------------------
# Task b): Build an LSTM model
# -------------------------------------------------------------
model = Sequential(
    [
        Embedding(
            input_dim=vocab_size, output_dim=16, input_length=max_length
        ),
        LSTM(32),
        Dense(1, activation="sigmoid"),
    ]
)

model.summary()

# -------------------------------------------------------------
# Task c): Compile the model
# -------------------------------------------------------------
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

# -------------------------------------------------------------
# Task d): Train the model and display training loss/accuracy
# -------------------------------------------------------------
history = model.fit(X_train, y_train, epochs=10, batch_size=2, verbose=1)

# -------------------------------------------------------------
# Task e): Predict the sentiment probability for each test review
# -------------------------------------------------------------
predicted_probabilities = model.predict(X_test)
print("\nPredicted Probabilities:\n", predicted_probabilities)

# -------------------------------------------------------------
# Task f): Display predicted probability and sentiment classification
# -------------------------------------------------------------
for review, prob in zip(test_reviews, predicted_probabilities):
  sentiment = "Positive" if prob[0] >= 0.5 else "Negative"
  print(
      f"Review: '{review}' | Probability: {prob[0]:.4f} | Sentiment:"
      f" {sentiment}"
  )

# -------------------------------------------------------------
# Task g): Explain how an LSTM helps capture long-term dependencies
# -------------------------------------------------------------
# Write these points in your exam if asked for a theoretical explanation:
#
# 1. Memory Cells & Gating Mechanism: LSTMs use specialized memory cells and
#    gates (input, forget, and output gates) to control the flow of information.
# 2. Preventing Vanishing Gradients: Unlike standard RNNs which struggle with
#    long sequences due to vanishing gradients, the forget gate allows the LSTM
#    to selectively retain or discard information over long time steps.
# 3. Context Retention: This architecture enables the network to remember
#    important context from earlier words in a sentence to influence predictions
#    made much later in the text sequence.

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 10, 16)            576       
                                                                 
 lstm (LSTM)                 (None, 32)                6272      
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 6881 (26.88 KB)
Trainable params: 6881 (26.88 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10
4/4 [==============================] - 2s 5ms/step - loss: 0.6945 - accuracy: 0.5000
Epoch 2/10
4/4 [==============================] - 0s 5ms/step - loss: 0.6914 - accuracy: 0.5000
Epoch 3/10
4/4 [==============================] - 0s 0s/step - loss: 0.6889 - accuracy: 0.50

In [4]:
#9 -> Electronic Device Review Classification Using a GRU Model
import numpy as np
from tensorflow.keras.layers import GRU, Dense, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Given dataset from the question
reviews = [
    "The smartphone battery lasts all day.",
    "The display is bright and the camera is excellent.",
    "This laptop is fast and works perfectly.",
    "The headphones have excellent sound quality.",
    "The smartphone overheats and the battery is poor.",
    "The laptop is slow and frequently crashes.",
    "The headphones stopped working after two days.",
    "The camera quality is disappointing.",
]

labels = [1, 1, 1, 1, 0, 0, 0, 0]

test_reviews = [
    "The laptop works fast and perfectly.",
    "The smartphone overheats and performs poorly.",
]

# -------------------------------------------------------------
# Task a): Tokenize and convert into padded sequences
# -------------------------------------------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts(reviews)

train_sequences = tokenizer.texts_to_sequences(reviews)
test_sequences = tokenizer.texts_to_sequences(test_reviews)

max_length = max(
    max(len(seq) for seq in train_sequences),
    max(len(seq) for seq in test_sequences),
)

X_train = pad_sequences(train_sequences, maxlen=max_length, padding="pre")
X_test = pad_sequences(test_sequences, maxlen=max_length, padding="pre")

y_train = np.array(labels)
vocab_size = len(tokenizer.word_index) + 1

# -------------------------------------------------------------
# Task b): Build a GRU model using an Embedding layer, a GRU layer, and a sigmoid output layer
# -------------------------------------------------------------
model = Sequential(
    [
        Embedding(
            input_dim=vocab_size, output_dim=16, input_length=max_length
        ),
        GRU(32),
        Dense(1, activation="sigmoid"),
    ]
)

model.summary()

# -------------------------------------------------------------
# Task c): Compile and train the model using binary cross-entropy loss
# -------------------------------------------------------------
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

# -------------------------------------------------------------
# Task d): Display the training accuracy and loss
# -------------------------------------------------------------
history = model.fit(X_train, y_train, epochs=10, batch_size=2, verbose=1)

# -------------------------------------------------------------
# Task e): Predict the sentiment probability for each test review
# -------------------------------------------------------------
predicted_probabilities = model.predict(X_test)
print("\nPredicted Probabilities:\n", predicted_probabilities)

# -------------------------------------------------------------
# Task f): Display the predicted probability and sentiment (Positive or Negative)
# -------------------------------------------------------------
for review, prob in zip(test_reviews, predicted_probabilities):
  sentiment = "Positive" if prob[0] >= 0.5 else "Negative"
  print(
      f"Review: '{review}' | Probability: {prob[0]:.4f} | Sentiment:"
      f" {sentiment}"
  )

# -------------------------------------------------------------
# Task g): Explain the difference between GRU and LSTM architectures
# -------------------------------------------------------------
# Write these points in your exam if asked for a theoretical explanation:
#
# 1. Fewer Gates: GRU (Gated Recurrent Unit) has two gates (Reset and Update)
#    whereas LSTM has three gates (Input, Forget, and Output).
# 2. No Separate Cell State: GRU combines the cell state and hidden state into a
#    single hidden state, making its architecture simpler.
# 3. Computational Efficiency: Due to fewer parameters and gates, GRU is faster
#    to train and requires less memory, while LSTM offers more fine-grained
#    control for extremely long sequences.

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 9, 16)             528       
                                                                 
 gru (GRU)                   (None, 32)                4800      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 5361 (20.94 KB)
Trainable params: 5361 (20.94 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/10
4/4 [==============================] - 2s 6ms/step - loss: 0.6949 - accuracy: 0.2500
Epoch 2/10
4/4 [==============================] - 0s 7ms/step - loss: 0.6908 - accuracy: 0.5000
Epoch 3/10
4/4 [==============================] - 0s 4ms/step - loss: 0.6879 - accuracy: 0.6

In [5]:
#10=> Comparative Text Classification Using TF-IDF, SVM, and Naive Bayes
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

# Given dataset from the question
messages = [
    "Your account has been selected for a free reward.",
    "Please reset your password using the official portal.",
    "Claim your cash prize immediately.",
    "Your support ticket has been resolved.",
    "Congratulations! You have won a free gift card.",
    "The scheduled maintenance will begin tonight.",
    "Earn money quickly by clicking this link.",
    "Your order has been shipped successfully.",
    "Urgent! Claim your exclusive bonus now.",
    "Please contact support regarding your account issue.",
    "Get a free voucher by responding to this message.",
    "Your appointment has been confirmed.",
]

labels = [
    "spam",
    "legitimate",
    "spam",
    "legitimate",
    "spam",
    "legitimate",
    "spam",
    "legitimate",
    "spam",
    "legitimate",
    "spam",
    "legitimate",
]

test_messages = [
    "Claim your free cash reward immediately.",
    "Your order confirmation has been sent.",
]

# -------------------------------------------------------------
# Task a): Split the dataset into training and testing sets
# -------------------------------------------------------------
X_train, X_test_split, y_train, y_test_split = train_test_split(
    messages, labels, test_size=0.25, random_state=42
)

# -------------------------------------------------------------
# Task b): Extract features using TfidfVectorizer
# -------------------------------------------------------------
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test_split)
X_external_tfidf = vectorizer.transform(test_messages)

# -------------------------------------------------------------
# Task c): Train an SVM classifier using SVC
# -------------------------------------------------------------
svm_model = SVC(kernel="linear")
svm_model.fit(X_train_tfidf, y_train)

# -------------------------------------------------------------
# Task d): Train a Multinomial Naive Bayes classifier
# -------------------------------------------------------------
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# -------------------------------------------------------------
# Task e): Predict the classes of the test messages using both classifiers
# -------------------------------------------------------------
svm_preds = svm_model.predict(X_external_tfidf)
nb_preds = nb_model.predict(X_external_tfidf)

print("SVM Predictions on Test Messages:", svm_preds)
print("Naive Bayes Predictions on Test Messages:", nb_preds)

# -------------------------------------------------------------
# Task f): Display predicted labels and class probabilities
# -------------------------------------------------------------
# SVM decision function (distance to hyperplane) or decision confidence
svm_decision = svm_model.decision_function(X_external_tfidf)
nb_probs = nb_model.predict_proba(X_external_tfidf)

print("\n--- SVM Results ---")
for msg, pred, score in zip(test_messages, svm_preds, svm_decision):
  print(f"Message: '{msg}' | Predicted: {pred} | Score: {score:.4f}")

print("\n--- Naive Bayes Results ---")
for msg, pred, prob in zip(test_messages, nb_preds, nb_probs):
  print(
      f"Message: '{msg}' | Predicted: {pred} | Probabilities (legit, spam):"
      f" {prob}"
  )

# -------------------------------------------------------------
# Task g): Evaluate and compare both classifiers (on internal split test set)
# -------------------------------------------------------------
y_pred_svm = svm_model.predict(X_test_tfidf)
y_pred_nb = nb_model.predict(X_test_tfidf)

print("\n--- SVM Evaluation Metrics ---")
print("Accuracy:", accuracy_score(y_test_split, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test_split, y_pred_svm))

print("\n--- Naive Bayes Evaluation Metrics ---")
print("Accuracy:", accuracy_score(y_test_split, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test_split, y_pred_nb))

# -------------------------------------------------------------
# Task h): Explain advantages and limitations of SVM and Naive Bayes
# -------------------------------------------------------------
# Write these points in your exam if asked for a theoretical explanation:
#
# 1. Support Vector Machine (SVM):
#    - Advantages: Highly effective in high-dimensional spaces (like text with TF-IDF); robust to overfitting.
#    - Limitations: Computationally intensive on very large datasets; sensitive to noise and missing feature scaling.
#
# 2. Multinomial Naive Bayes (NB):
#    - Advantages: Extremely fast, computationally lightweight, requires very little training data, performs exceptionally well for text classification (spam filtering).
#    - Limitations: Makes a strong "naive" assumption that all features (words) are independent, which is rarely true in natural language.

SVM Predictions on Test Messages: ['spam' 'legitimate']
Naive Bayes Predictions on Test Messages: ['spam' 'legitimate']

--- SVM Results ---
Message: 'Claim your free cash reward immediately.' | Predicted: spam | Score: 0.9459
Message: 'Your order confirmation has been sent.' | Predicted: legitimate | Score: -0.9538

--- Naive Bayes Results ---
Message: 'Claim your free cash reward immediately.' | Predicted: spam | Probabilities (legit, spam): [0.35194741 0.64805259]
Message: 'Your order confirmation has been sent.' | Predicted: legitimate | Probabilities (legit, spam): [0.76982602 0.23017398]

--- SVM Evaluation Metrics ---
Accuracy: 0.6666666666666666
Confusion Matrix:
 [[1 0]
 [1 1]]

--- Naive Bayes Evaluation Metrics ---
Accuracy: 0.6666666666666666
Confusion Matrix:
 [[1 0]
 [1 1]]
